# Module 17 - Retrieval-augmented generation

Use this notebook after `tests/test_rag.py` is passing. The notebook builds a small RAG pipeline in layers: chunks, embeddings, vector search, prompt assembly, retrieval, then generation through a backend.

The deliverable is the RAG postmortem: what you indexed, what retrieved well, where retrieval failed, and what you would improve before using this as part of the assistant.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from g2c.inference import Backend, BackendInfo, InferenceResult, load_prodlm_backend, prodlm_manifest_exists
from g2c.notebook_extras.sampling import printable
from g2c.rag import (
    DEFAULT_INSTRUCTION,
    DEFAULT_OLLAMA_EMBED_MODEL,
    DEFAULT_SYSTEM,
    Chunk,
    DenseRetriever,
    HashEmbedder,
    NumpyVectorStore,
    OllamaEmbedder,
    RAGPipeline,
    RetrievedChunk,
    assemble_rag_prompt,
    chunk_text,
    cosine_similarity,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

/Users/colkitt/sith/toys/courses/g2c


Run the RAG tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 17 TODOs in `g2c/rag/`.

In [2]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_rag.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 17 RAG tests are not passing yet."

........................................................................ [ 50%]
......................................................................   [100%]



## Display helpers

In [3]:
def short(text: Any, limit: int = 220) -> str:
    rendered = printable(str(text)).replace("\n", "\\n")
    if len(rendered) <= limit:
        return rendered
    return rendered[: limit - 3] + "..."


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    def cell(value: Any) -> str:
        text = str(value).replace("|", "\\|").replace("\n", "<br>")
        return text

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(cell(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join([header, sep, *body])


def show_chunks(chunks: list[Chunk], *, limit: int = 8) -> None:
    rows = []
    for i, chunk in enumerate(chunks[:limit], start=1):
        rows.append(
            {
                "i": i,
                "source": chunk.source,
                "span": f"{chunk.start}:{chunk.end}",
                "chars": len(chunk.text),
                "text": short(chunk.text, 120),
            }
        )
    display(Markdown(markdown_table(rows, ["i", "source", "span", "chars", "text"])))
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")


def show_retrieved(results: list[RetrievedChunk]) -> None:
    rows = []
    for r in results:
        rows.append(
            {
                "rank": r.rank,
                "score": f"{r.score:.3f}",
                "source": r.chunk.source,
                "text": short(r.chunk.text, 180),
            }
        )
    display(Markdown(markdown_table(rows, ["rank", "score", "source", "text"])))


def show_rag_answer(answer) -> None:
    print(printable(answer.answer))
    print()
    print("Sources:")
    show_retrieved(answer.retrieved)
    print("metadata:", answer.metadata)


## A tiny corpus

The first pass uses a deliberately small in-memory corpus so the moving parts are visible. Later cells swap in course docs and optional Ollama embeddings.

In [4]:
documents = {
    "cities.md": (
        "Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\n"
        "Paris is the capital of France. The Seine river runs through the city.\n\n"
        "Tokyo is the capital of Japan and one of the largest metropolitan areas in the world."
    ),
    "fruit.md": (
        "Bananas are yellow fruit rich in potassium.\n\n"
        "Apples grow on trees and can be red, green, or yellow.\n\n"
        "Oranges are citrus fruit and are often used for juice."
    ),
    "course.md": (
        "Module 16 introduces inference backends and ProdLM.\n\n"
        "Module 17 introduces retrieval augmented generation over an external corpus.\n\n"
        "Module 18 adds tools so the assistant can act outside text generation."
    ),
}

for source, text in documents.items():
    print(source, len(text), "chars")


cities.md 248 chars
fruit.md 155 chars
course.md 201 chars


## Exercise 1 - Chunk documents

`chunk_text` turns each source document into overlapping retrievable slices. The key parameters are `chunk_size` and `chunk_overlap`.

In [5]:
all_chunks: list[Chunk] = []
for source, text in documents.items():
    all_chunks.extend(
        chunk_text(
            text,
            source=source,
            chunk_size=120,
            chunk_overlap=30,
            metadata={"corpus": "toy"},
        )
    )

print("chunks:", len(all_chunks))
show_chunks(all_chunks)

chunks: 7


| i | source | span | chars | text |
| --- | --- | --- | --- | --- |
| 1 | cities.md | 0:120 | 120 | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of ... |
| 2 | cities.md | 90:210 | 120 | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of ... |
| 3 | cities.md | 180:248 | 68 | tal of Japan and one of the largest metropolitan areas in the world. |
| 4 | fruit.md | 0:120 | 120 | Bananas are yellow fruit rich in potassium.\n\nApples grow on trees and can be red, green, or yellow.\n\nOranges are ... |
| 5 | fruit.md | 90:155 | 65 | r yellow.\n\nOranges are citrus fruit and are often used for juice. |
| 6 | course.md | 0:120 | 120 | Module 16 introduces inference backends and ProdLM.\n\nModule 17 introduces retrieval augmented generation over an ex... |
| 7 | course.md | 90:201 | 111 | ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation. |

In [6]:
# Inspect the overlap directly on the first document.
city_chunks = [chunk for chunk in all_chunks if chunk.source == "cities.md"]
show_chunks(city_chunks)

if len(city_chunks) >= 2:
    previous = city_chunks[0]
    current = city_chunks[1]
    overlap = previous.end - current.start
    print("actual overlap:", overlap)
    print("previous suffix:", repr(previous.text[-overlap:]))
    print("current prefix:", repr(current.text[:overlap]))

| i | source | span | chars | text |
| --- | --- | --- | --- | --- |
| 1 | cities.md | 0:120 | 120 | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of ... |
| 2 | cities.md | 90:210 | 120 | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of ... |
| 3 | cities.md | 180:248 | 68 | tal of Japan and one of the largest metropolitan areas in the world. |

actual overlap: 30
previous suffix: '\nParis is the capital of Franc'
current prefix: '\nParis is the capital of Franc'


Try a few chunk sizes and watch the number of chunks change. Smaller chunks retrieve more specifically; larger chunks carry more context but blur the embedding.

In [7]:
chunk_size_sweep = []
for size in [60, 100, 160, 240]:
    overlap = max(0, size // 5)
    n_chunks = sum(
        len(chunk_text(text, source=source, chunk_size=size, chunk_overlap=overlap))
        for source, text in documents.items()
    )
    chunk_size_sweep.append({"chunk_size": size, "overlap": overlap, "chunks": n_chunks})

display(Markdown(markdown_table(chunk_size_sweep, ["chunk_size", "overlap", "chunks"])))

| chunk_size | overlap | chunks |
| --- | --- | --- |
| 60 | 12 | 12 |
| 100 | 20 | 8 |
| 160 | 32 | 5 |
| 240 | 48 | 4 |

## Exercise 2 - Hash embeddings

`HashEmbedder` is a toy lexical embedder. It is not semantic, but it makes the same vector-store math work without an external embedding model.

In [8]:
hash_embedder = HashEmbedder(dim=512, ngram_range=(3, 5), seed=0)
chunk_vectors = hash_embedder.embed([chunk.text for chunk in all_chunks])

print("vectors:", chunk_vectors.shape, chunk_vectors.dtype)
print("first five row norms:", np.linalg.norm(chunk_vectors[:5], axis=1))

vectors: (7, 512) float32
first five row norms: [0.99999994 1.         1.         1.         1.        ]


In [10]:
probe_texts = [
    "Madrid is the capital of Spain.",
    "Madrid is Spain's capital city.",
    "What is the capital of Spain?",
    "Bananas are a yellow fruit.",
]
probe_vectors = hash_embedder.embed(probe_texts)

rows = []
for i, a in enumerate(probe_texts):
    for j, b in enumerate(probe_texts):
        if j <= i:
            continue
        rows.append(
            {
                "a": short(a, 45),
                "b": short(b, 45),
                "cosine": f"{cosine_similarity(probe_vectors[i], probe_vectors[j]):.3f}",
            }
        )

display(Markdown(markdown_table(rows, ["a", "b", "cosine"])))

| a | b | cosine |
| --- | --- | --- |
| Madrid is the capital of Spain. | Madrid is Spain's capital city. | 0.667 |
| Madrid is the capital of Spain. | What is the capital of Spain? | 0.811 |
| Madrid is the capital of Spain. | Bananas are a yellow fruit. | 0.127 |
| Madrid is Spain's capital city. | What is the capital of Spain? | 0.506 |
| Madrid is Spain's capital city. | Bananas are a yellow fruit. | 0.123 |
| What is the capital of Spain? | Bananas are a yellow fruit. | 0.144 |

## Exercise 3 - Build and search a vector store

A flat vector store keeps chunks and vectors in matching order. Search is one dot product per chunk plus a top-k selection.

In [11]:
store = NumpyVectorStore(dim=hash_embedder.dim)
store.add(all_chunks, chunk_vectors)
print(store)

query = "What city is the capital of Spain?"
query_vector = hash_embedder.embed([query])[0]
raw_results = store.search(query_vector, k=4)

rows = []
for rank, (chunk, score) in enumerate(raw_results, start=1):
    rows.append(
        {
            "rank": rank,
            "score": f"{score:.3f}",
            "source": chunk.source,
            "text": short(chunk.text, 180),
        }
    )

display(Markdown(markdown_table(rows, ["rank", "score", "source", "text"])))

NumpyVectorStore(dim=512, n=7)


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.735 | cities.md | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of Franc |
| 2 | 0.630 | cities.md | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of the la |
| 3 | 0.336 | cities.md | tal of Japan and one of the largest metropolitan areas in the world. |
| 4 | 0.259 | course.md | ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation. |

In [12]:
query_examples = [
    "What city is the capital of Spain?",
    "Which module adds tools?",
    "What fruit has potassium?",
    "Which river runs through Paris?",
]

for query in query_examples:
    qvec = hash_embedder.embed([query])[0]
    results = store.search(qvec, k=2)
    print("=" * 80)
    print(query)
    for rank, (chunk, score) in enumerate(results, start=1):
        print(f"[{rank}] score={score:.3f} source={chunk.source} text={short(chunk.text, 140)}")

What city is the capital of Spain?
[1] score=0.735 source=cities.md text=Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of Franc
[2] score=0.630 source=cities.md text=\nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of the la
Which module adds tools?
[1] score=0.391 source=course.md text=ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation.
[2] score=0.283 source=course.md text=Module 16 introduces inference backends and ProdLM.\n\nModule 17 introduces retrieval augmented generation over an externa
What fruit has potassium?
[1] score=0.417 source=fruit.md text=Bananas are yellow fruit rich in potassium.\n\nApples grow on trees and can be red, green, or yellow.\n\nOranges are citrus 
[2] score=0.346 source=fruit.md text=r yellow.\n\nOranges are citrus fruit and are often used for juice.
Which river

## Exercise 4 - Use the retriever abstraction

`DenseRetriever` wires the embedder and vector store into the interface later assistant modules expect: `retrieve(query, k)` returns ranked chunks.

In [13]:
retriever = DenseRetriever(hash_embedder, store)
retrieved = retriever.retrieve("Which module adds tools to the assistant?", k=3)
show_retrieved(retrieved)

| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.511 | course.md | ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation. |
| 2 | 0.332 | cities.md | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of Franc |
| 3 | 0.331 | cities.md | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of the la |

## Exercise 5 - Assemble a RAG prompt

Prompt assembly is the augmentation step. The retrieved chunks are numbered so the model can cite them.

In [14]:
question = "Which module adds tools to the assistant?"
prompt = assemble_rag_prompt(question, retrieved)
print(prompt.text)

You are a helpful assistant. Answer the user's question using ONLY the context below. Refer to context items by their bracket numbers, e.g. [1], [2].

Context:
[1] (source: course.md)
ted generation over an external corpus.

Module 18 adds tools so the assistant can act outside text generation.

[2] (source: cities.md)
Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.

Paris is the capital of Franc

[3] (source: cities.md)

Paris is the capital of France. The Seine river runs through the city.

Tokyo is the capital of Japan and one of the la

Question: Which module adds tools to the assistant?

If the context does not contain the information needed to answer the question, reply with: "I don't know based on the provided context." Do not invent facts that are not in the context.


In [15]:
empty_prompt = assemble_rag_prompt("What is the population of Pluto?", [])
print(empty_prompt.text)

You are a helpful assistant. Answer the user's question using ONLY the context below. Refer to context items by their bracket numbers, e.g. [1], [2].

Context:

Question: What is the population of Pluto?

If the context does not contain the information needed to answer the question, reply with: "I don't know based on the provided context." Do not invent facts that are not in the context.


## Exercise 6 - End-to-end RAG with a fake backend

Before calling a live model, use a deterministic backend. This confirms retrieval and prompt assembly are wired correctly.

In [16]:
class FakeBackend(Backend):
    def __init__(self, scripted_answer: str = "MOCK ANSWER") -> None:
        self.scripted_answer = scripted_answer
        self.last_prompt: str | None = None
        self.last_kwargs: dict[str, Any] | None = None
        self._info = BackendInfo(name="fake", model_id="fake-rag")

    @property
    def info(self) -> BackendInfo:
        return self._info

    def complete(
        self,
        prompt: str,
        *,
        max_new_tokens: int = 128,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> InferenceResult:
        self.last_prompt = prompt
        self.last_kwargs = {
            "max_new_tokens": max_new_tokens,
            "temperature": temperature,
            "top_k": top_k,
            "top_p": top_p,
        }
        return InferenceResult(
            prompt=prompt,
            completion=self.scripted_answer,
            prompt_tokens=len(prompt.split()),
            completion_tokens=len(self.scripted_answer.split()),
            latency_ms=1.0,
            backend=self._info,
        )

In [17]:
fake_backend = FakeBackend("Module 18 adds tools to the assistant [1].")
fake_pipeline = RAGPipeline(retriever, fake_backend)
fake_answer = fake_pipeline.answer("Which module adds tools to the assistant?", k=3)
show_rag_answer(fake_answer)

print("\nPrompt sent to backend:")
print(fake_backend.last_prompt)


Module 18 adds tools to the assistant [1].

Sources:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.511 | course.md | ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation. |
| 2 | 0.332 | cities.md | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of Franc |
| 3 | 0.331 | cities.md | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of the la |

metadata: {'k': 3, 'n_retrieved': 3, 'backend_name': 'fake', 'backend_model_id': 'fake-rag'}

Prompt sent to backend:
You are a helpful assistant. Answer the user's question using ONLY the context below. Refer to context items by their bracket numbers, e.g. [1], [2].

Context:
[1] (source: course.md)
ted generation over an external corpus.

Module 18 adds tools so the assistant can act outside text generation.

[2] (source: cities.md)
Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.

Paris is the capital of Franc

[3] (source: cities.md)

Paris is the capital of France. The Seine river runs through the city.

Tokyo is the capital of Japan and one of the la

Question: Which module adds tools to the assistant?

If the context does not contain the information needed to answer the question, reply with: "I don't know based on the provided context." Do not invent facts that are not in the context.


## Exercise 7 - Index course module docs

The toy corpus is enough to understand the mechanics. A useful RAG pipeline needs a corpus you care about. This cell indexes the course module markdown files with the hash embedder.

In [18]:
DOC_GLOB = "*.md"
DOC_LIMIT = 24
COURSE_CHUNK_SIZE = 1400
COURSE_CHUNK_OVERLAP = 200

module_dir = repo_root / "docs" / "modules"
module_paths = sorted(module_dir.glob(DOC_GLOB))[:DOC_LIMIT]
print("documents:", len(module_paths))
for path in module_paths[:8]:
    print(" ", path.relative_to(repo_root))

course_chunks: list[Chunk] = []
start = time.perf_counter()
for path in module_paths:
    text = path.read_text(encoding="utf-8")
    course_chunks.extend(
        chunk_text(
            text,
            source=str(path.relative_to(repo_root)),
            chunk_size=COURSE_CHUNK_SIZE,
            chunk_overlap=COURSE_CHUNK_OVERLAP,
            metadata={"kind": "course-doc"},
        )
    )
elapsed = time.perf_counter() - start
print(f"chunks: {len(course_chunks):,} built in {elapsed:.2f}s")
show_chunks(course_chunks, limit=6)

documents: 23
  docs/modules/00-prerequisite-review.md
  docs/modules/01-autodiff.md
  docs/modules/02-tensors.md
  docs/modules/03-nn.md
  docs/modules/03b-training.md
  docs/modules/04-tokenizer.md
  docs/modules/05-embeddings.md
  docs/modules/06-language-models.md
chunks: 520 built in 0.01s


| i | source | span | chars | text |
| --- | --- | --- | --- | --- |
| 1 | docs/modules/00-prerequisite-review.md | 0:1400 | 1400 | # Module 00 — Prerequisite review\n\n> **Question this module answers:** *What do I need back in cache before buildin... |
| 2 | docs/modules/00-prerequisite-review.md | 1200:2600 | 1400 | sampling.\n### Computer science\n\n- **Functions and composition.** The whole course treats models as large composed ... |
| 3 | docs/modules/00-prerequisite-review.md | 2400:3800 | 1400 | g if derivatives, the chain rule, and "a computation as a graph" are already close at hand. Module 02 immediately mov... |
| 4 | docs/modules/00-prerequisite-review.md | 3600:5000 | 1400 | lains why logits, softmax, cross-entropy, and perplexity are the right language for prediction.\n- **ML workflow** ke... |
| 5 | docs/modules/00-prerequisite-review.md | 4800:6200 | 1400 | C)`: one row for each token ID, and one `C`-dimensional vector in each row. If token IDs have shape `(B, T)`, looking... |
| 6 | docs/modules/00-prerequisite-review.md | 6000:7400 | 1400 | ddings, attention, logits, and softmax; writing them down is the fastest way to catch a mistaken transpose, missing b... |

... 514 more chunks


In [19]:
course_embedder = HashEmbedder(dim=1024, ngram_range=(3, 5), seed=17)
start = time.perf_counter()
course_vectors = course_embedder.embed([chunk.text for chunk in course_chunks])
course_embed_seconds = time.perf_counter() - start

course_store = NumpyVectorStore(dim=course_embedder.dim)
course_store.add(course_chunks, course_vectors)
course_retriever = DenseRetriever(course_embedder, course_store)

print(course_store)
print(f"hash embedding wall time: {course_embed_seconds:.2f}s")

NumpyVectorStore(dim=1024, n=520)
hash embedding wall time: 1.19s


In [20]:
course_questions = [
    "Which module introduces retrieval augmented generation?",
    "What does Module 16 build?",
    "Which module adds tools?",
    "What is the deliverable for the capstone?",
]

for question in course_questions:
    print("=" * 80)
    print(question)
    show_retrieved(course_retriever.retrieve(question, k=3))

Which module introduces retrieval augmented generation?


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.493 | docs/modules/17-rag.md | cale.\n- **Anthropic, "Contextual retrieval" blog post (Sep 2024).** A practical exposition of "prepend a chunk-specific summary to each chunk before embedding" as a retrieval-q... |
| 2 | 0.440 | docs/modules/11-sampling.md | *Fan, Lewis, Dauphin, "Hierarchical Neural Story Generation" (2018).** The top-k paper, in the context of story generation. Older but worth reading; introduces the diversity-vs-... |
| 3 | 0.438 | docs/modules/17-rag.md | .\n\n- **`Chunk` is frozen but `Chunk.metadata` is not.** `dataclass(frozen=True)` freezes attribute assignment, not nested objects. `c.text = "x"` raises; `c.metadata["k"] = 1`... |

What does Module 16 build?


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.350 | docs/modules/17-rag.md | # Module 17 — Retrieval-augmented generation\n\n> **Question this module answers:** *How can the model use external knowledge it doesn't have memorized?*\n\n![Hero](17-rag/Modul... |
| 2 | 0.329 | docs/modules/05-embeddings.md | # Module 05 — Embeddings and positions\n\n> **Question this module answers:** *How do discrete symbols become meaning-like vectors?*\n\n![From token IDs to meaning-like vectors:... |
| 3 | 0.327 | docs/modules/16-inference.md | # Module 16 — Inference backends and production models\n\n> **Question this module answers:** *How do we get from "I built it" to "I can use it"?*\n\n![Module 16 summary diagram... |

Which module adds tools?


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.328 | docs/modules/10-tinyllm.md | # Module 10 — Milestone: TinyLLM\n\n> **Question this module answers:** *Can I build a language model using the tools we learned?*\n\n![Pretraining the tiny GPT end-to-end: a ra... |
| 2 | 0.314 | docs/modules/19-agent.md | # Module 19 — Agent loops\n\n> **Question this module answers:** *How do we make the model pursue goals?*\n\n![Agent loop wrapped around a model backend and tool registry](19-ag... |
| 3 | 0.301 | docs/modules/15-evaluation.md | # Module 15 — Hallucination and evaluation\n\n> **Question this module answers:** *Why does the model confidently invent things, and how do we measure it?*\n\n![Hero](15-evaluat... |

What is the deliverable for the capstone?


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.470 | docs/modules/20-capstone.md | re unambiguously. Two channels, two purposes.\n\n- **The eval gate runs in seconds and refuses regressions.** The point is not to measure how good the assistant is in absolute t... |
| 2 | 0.461 | docs/modules/20-capstone.md |  turn 8 references something from turn 1. Does the model lose the thread? At what `max_history` does the reference work reliably? Where would summarization (compress old turns i... |
| 3 | 0.436 | docs/modules/17-rag.md | now" guard. Without it, models hallucinate when context is insufficient. With it, instruction-tuned models will (more often than not) actually abstain.\n\nNote what's *not* in t... |

## Exercise 8 - Optional semantic embeddings with Ollama

Hash embeddings are lexical. If you ran `./prodlm.sh`, you should also have `nomic-embed-text` available through Ollama. Set `RUN_OLLAMA_EMBEDDINGS = True` to build a semantic index over the same chunks.

In [21]:
RUN_OLLAMA_EMBEDDINGS = False
OLLAMA_EMBED_MODEL = DEFAULT_OLLAMA_EMBED_MODEL
OLLAMA_EMBED_DIM = 768
OLLAMA_DOC_LIMIT = 12

ollama_retriever = None
if RUN_OLLAMA_EMBEDDINGS:
    ollama_embedder = OllamaEmbedder(OLLAMA_EMBED_MODEL, dim=OLLAMA_EMBED_DIM)
    ollama_chunks = course_chunks[:OLLAMA_DOC_LIMIT]
    start = time.perf_counter()
    ollama_vectors = ollama_embedder.embed([chunk.text for chunk in ollama_chunks])
    elapsed = time.perf_counter() - start
    ollama_store = NumpyVectorStore(dim=ollama_embedder.dim)
    ollama_store.add(ollama_chunks, ollama_vectors)
    ollama_retriever = DenseRetriever(ollama_embedder, ollama_store)
    print(f"embedded {len(ollama_chunks)} chunks in {elapsed:.2f}s")
else:
    print("Skipping Ollama embeddings. Set RUN_OLLAMA_EMBEDDINGS = True after running ./prodlm.sh.")

Skipping Ollama embeddings. Set RUN_OLLAMA_EMBEDDINGS = True after running ./prodlm.sh.


In [22]:
semantic_question = "What part of the course lets the assistant use information outside its weights?"

print("Hash retriever:")
show_retrieved(course_retriever.retrieve(semantic_question, k=3))

if ollama_retriever is not None:
    print("Ollama semantic retriever:")
    show_retrieved(ollama_retriever.retrieve(semantic_question, k=3))

Hash retriever:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.534 | docs/modules/20-capstone.md |  end of the turn; append user + final-answer messages to Conversation. The model sees BOTH on every backend call inside step 3: the rendered conversation as part of the user_mes... |
| 2 | 0.513 | docs/modules/17-rag.md | ge collection of information in the form of a corpus (not necessarily the same corpus used in pretraining) and what information it selectively curates at [[16-inferance]] time t... |
| 3 | 0.510 | docs/modules/20-capstone.md |  turn 8 references something from turn 1. Does the model lose the thread? At what `max_history` does the reference work reliably? Where would summarization (compress old turns i... |

## Exercise 9 - Optional live RAG with ProdLM

This turns the retrieved chunks into a real answer. The notebook defaults to ProdLM if a manifest exists, but all cells are safe to skip when Ollama is not running.

In [23]:
RUN_PRODLM_RAG = prodlm_manifest_exists(repo_root=repo_root)
PRODLM_MODEL_ID = None  # set to an Ollama tag to override configured ProdLM
USE_SEMANTIC_RETRIEVER_IF_AVAILABLE = True

rag_backend = None
if RUN_PRODLM_RAG:
    rag_backend = load_prodlm_backend(repo_root=repo_root, model_id=PRODLM_MODEL_ID, required=False)
    print("loaded:", rag_backend.info)
else:
    print("ProdLM is not configured. Run ./prodlm.sh, then set RUN_PRODLM_RAG = True.")

live_retriever = ollama_retriever if (USE_SEMANTIC_RETRIEVER_IF_AVAILABLE and ollama_retriever is not None) else course_retriever
print("retriever:", live_retriever)

loaded: BackendInfo(name='prodlm', model_id='llama3.2:3b', extra={'base_url': 'http://localhost:11434', 'configured_name': 'ProdLM'})
retriever: DenseRetriever(embedder=HashEmbedder(dim=1024, ngram_range=(3, 5), seed=17), store=NumpyVectorStore(dim=1024, n=520))


In [24]:
live_pipeline = None
if rag_backend is not None:
    live_pipeline = RAGPipeline(live_retriever, rag_backend)
else:
    print("No live backend loaded.")


def ask_rag(question: str, *, k: int = 4, max_new_tokens: int = 220):
    if live_pipeline is None:
        print("No live RAG pipeline loaded.")
        return None
    try:
        answer = live_pipeline.answer(
            question,
            k=k,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
        )
    except Exception as exc:
        print(f"RAG call failed: {type(exc).__name__}: {exc}")
        return None
    show_rag_answer(answer)
    return answer

In [25]:
live_question = "What does Module 17 teach, and why does retrieval quality matter?"
live_answer = ask_rag(live_question, k=4)

Based on the provided context, Module 17 teaches a recipe for modern dense retrieval using an encoder that embeds both queries and passages via inner product. It also discusses various secondary papers related to retrieval quality.

Retrieval quality matters because it dominates RAG (Reinforcement Augmented Generation) quality. Improving the retriever can lead to more significant improvements in overall performance than improving the model itself. This is evident from the emphasis on testing and evaluating the retriever's performance, as well as the discussion of retrieval-quality metrics.

In particular, retrieval quality is crucial because it allows the model to effectively retrieve relevant chunks for generation, which is essential for tasks like RAG. Without high-quality retrieval, the model may struggle to generate coherent and accurate responses.

Therefore, the answer to the question is that Module 17 teaches a recipe for modern dense retrieval, and retrieval quality matters sig

| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.437 | docs/modules/17-rag.md | encoder, embed both queries and passages, retrieve by inner product" recipe that defines modern dense retrieval. Read §3 (training) and §4 (results). Skim §5 (analysis).\n- **Ro... |
| 2 | 0.432 | docs/modules/17-rag.md | is going.\n\n## Deliverable checklist\n\n- [ ] All tests in `tests/test_rag.py` pass\n- [ ] Ollama running with at least one embedding model pulled. `ollama list` shows `nomic-e... |
| 3 | 0.426 | docs/modules/17-rag.md | cale.\n- **Anthropic, "Contextual retrieval" blog post (Sep 2024).** A practical exposition of "prepend a chunk-specific summary to each chunk before embedding" as a retrieval-q... |
| 4 | 0.421 | docs/modules/20-capstone.md | e cases (arithmetic, file reading, Python execution, a couple of corpus questions). Save them in `notebooks/20-eval-cases.py`. Run `run_evaluation(assistant, cases)` and report ... |

metadata: {'k': 4, 'n_retrieved': 4, 'backend_name': 'prodlm', 'backend_model_id': 'llama3.2:3b'}


## Exercise 10 - Failure-mode probes

Good RAG work includes negative examples. You want to know when the model refuses, when retrieval misses, and when the generator invents facts anyway.

In [26]:
failure_questions = [
    {"bucket": "answerable", "question": "Which module adds tools to the assistant?"},
    {"bucket": "paraphrase", "question": "Where in the course does the model get external memory?"},
    {"bucket": "unanswerable", "question": "What is the population of Pluto according to these course docs?"},
]

probe_rows = []
for item in failure_questions:
    retrieved = live_retriever.retrieve(item["question"], k=3)
    probe_rows.append(
        {
            "bucket": item["bucket"],
            "question": item["question"],
            "top source": retrieved[0].chunk.source if retrieved else "",
            "top score": f"{retrieved[0].score:.3f}" if retrieved else "",
            "top text": short(retrieved[0].chunk.text, 130) if retrieved else "",
        }
    )

display(Markdown(markdown_table(probe_rows, ["bucket", "question", "top source", "top score", "top text"])))

| bucket | question | top source | top score | top text |
| --- | --- | --- | --- | --- |
| answerable | Which module adds tools to the assistant? | docs/modules/18-tools.md | 0.434 | e harness represents a separate type of external action. For example there might be a `web_search` tool, a `schedule` tool and ... |
| paraphrase | Where in the course does the model get external memory? | docs/modules/20-capstone.md | 0.515 |  takes a `query` argument, calls the retriever, and returns formatted chunks. Register it in the assistant's tool registry. Dis... |
| unanswerable | What is the population of Pluto according to these course docs? | docs/modules/03-nn.md | 0.457 | function plus a way to fit it to data.** Everything else — depth, layer types, optimizers, schedulers, regularization — is vari... |

In [27]:
RUN_FAILURE_PROBES_WITH_MODEL = True

if RUN_FAILURE_PROBES_WITH_MODEL:
    for item in failure_questions:
        print("=" * 80)
        print(item["bucket"], "-", item["question"])
        ask_rag(item["question"], k=3, max_new_tokens=180)
else:
    print("Set RUN_FAILURE_PROBES_WITH_MODEL = True to run the live answer probes.")

answerable - Which module adds tools to the assistant?
Based on the context, I can see that [1] mentions that each tool in the harness is made up of two components: a **Tool specification** and a **Tool callable**.

It also states that "The model doesn't have to learn the individual tools ahead of time... All we have to do to add a new tool is conform to the specification with enough descriptiveness that the model can infer at prompt time."

However, it does not explicitly state which module adds tools to the assistant.

Sources:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.434 | docs/modules/18-tools.md | e harness represents a separate type of external action. For example there might be a `web_search` tool, a `schedule` tool and a `run_python` tool.\n\nEach tool in the harness i... |
| 2 | 0.431 | docs/modules/20-capstone.md | g step, then the ReAct loop with `backend.complete` calls (Module 16) and `dispatch_tool_call` against the registry (Module 18). (6) RECORD USER + ASSISTANT in conversation, ret... |
| 3 | 0.431 | docs/modules/20-capstone.md | # Module 20 — Capstone: a tiny ChatGPT\n\n> **Question this module answers:** *Can I integrate everything?*\n\n![Module 20 on one page: a five-panel circus map of the capstone a... |

metadata: {'k': 3, 'n_retrieved': 3, 'backend_name': 'prodlm', 'backend_model_id': 'llama3.2:3b'}
paraphrase - Where in the course does the model get external memory?
Based on the provided context, I will answer your questions step by step.

You want me to compare prefix-style and tool-style RAGs. 

To do this, let's look at [1]. It says that the model has to *decide* when to call `search` and disables prefix-style RAG (`rag_enabled=False`). This means that the model now uses a tool-style RAG.

Now, let's compare prefix-style and tool-style RAGs:

Prefix-style RAGs are better when:
- The answer is not in the parametric memory.
- The model needs to search for the correct response.

Tool-style RAGs are better when:
- The answer is in the parametric memory.
- The model can use the tool-style RAG to quickly retrieve the correct response.

In general, prefix-style RAGs are useful when the model needs to search for

Sources:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.515 | docs/modules/20-capstone.md |  takes a `query` argument, calls the retriever, and returns formatted chunks. Register it in the assistant's tool registry. Disable prefix-style RAG (`rag_enabled=False`). Now t... |
| 2 | 0.506 | docs/modules/11-sampling.md | ock]]) \n	* At least one trained model from [[10-tinyllm]] notebook (`ShakespeareLM`, `StoryLM`, or `TinyLLM`), or run `./baselm.sh` and explicitly load `BaseLM` for comparison\... |
| 3 | 0.506 | docs/modules/13-sft.md |  They showed that **1000 carefully-curated SFT examples** produces a chat model nearly indistinguishable from tens of thousands of crowd-sourced examples. The active ingredient ... |

metadata: {'k': 3, 'n_retrieved': 3, 'backend_name': 'prodlm', 'backend_model_id': 'llama3.2:3b'}
unanswerable - What is the population of Pluto according to these course docs?
I don't know based on the provided context. There is no mention of Pluto or its population in the given text. The context appears to be related to neural networks and machine learning, with topics such as modules, training loops, autodiff, and optimization algorithms discussed.

Sources:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.457 | docs/modules/03-nn.md | function plus a way to fit it to data.** Everything else — depth, layer types, optimizers, schedulers, regularization — is variation on this theme. If you can write a 2-layer ML... |
| 2 | 0.452 | docs/modules/11-sampling.md | ock]]) \n	* At least one trained model from [[10-tinyllm]] notebook (`ShakespeareLM`, `StoryLM`, or `TinyLLM`), or run `./baselm.sh` and explicitly load `BaseLM` for comparison\... |
| 3 | 0.452 | docs/modules/01-autodiff.md | # Module 01 — Scalar autodiff\n\n> **Question this module answers:** *How does the model learn?*\n\n![Training loop summary: parameters → forward pass → loss → backward pass (au... |

metadata: {'k': 3, 'n_retrieved': 3, 'backend_name': 'prodlm', 'backend_model_id': 'llama3.2:3b'}


## Exercise 11 - Persist an index extension

The course package keeps `NumpyVectorStore` in memory, but persisting an index is a natural extension. This cell gives a small starter shape you can adapt for the postmortem or an optional script.

In [ ]:
def save_store_snapshot(store: NumpyVectorStore, path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    np.save(path / "vectors.npy", store.vectors)
    chunks_json = [
        {
            "text": chunk.text,
            "source": chunk.source,
            "start": chunk.start,
            "end": chunk.end,
            "metadata": chunk.metadata,
        }
        for chunk in store.chunks
    ]
    (path / "chunks.json").write_text(json.dumps(chunks_json, indent=2), encoding="utf-8")


def load_store_snapshot(path: Path, *, dim: int) -> NumpyVectorStore:
    vectors = np.load(path / "vectors.npy")
    chunks_data = json.loads((path / "chunks.json").read_text(encoding="utf-8"))
    chunks = [Chunk(**item) for item in chunks_data]
    store = NumpyVectorStore(dim=dim)
    store.add(chunks, vectors)
    return store

snapshot_dir = repo_root / "data" / "module17-rag" / "hash-index-snapshot"
save_store_snapshot(course_store, snapshot_dir)
round_trip_store = load_store_snapshot(snapshot_dir, dim=course_embedder.dim)
print(round_trip_store)
print("same vectors:", np.allclose(course_store.vectors, round_trip_store.vectors))
print("same first chunk:", course_store.chunks[0] == round_trip_store.chunks[0])

## Postmortem notes

Write `docs/rag-postmortem.md` in 3-4 paragraphs. Cover:

- What you indexed: corpus, chunk size, overlap, embedder, vector count.
- What worked: question types where retrieval reliably surfaced the right chunk.
- Where it broke: chunking, embedding, retrieval, prompt, or model refusal/hallucination.
- What you would build next: hybrid retrieval, re-ranking, smarter chunking, larger embedder, or a better eval set.